# 12 Robust validation

This milestone tests whether the 11-stage strict pre-protection result is stable across repeated trajectory-level splits, model choices, class balancing, minimum support, confusion pairs, and the A/B/C sensitivity groups. The four windows are evaluated on one matched cohort so that window comparisons use the same trajectories and the same split assignments.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore', message='y_pred contains classes not in y_true')
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.multi_accident_features import FIRST_VERSION_CLASSES, input_feature_groups
from src.temporal_diagnosability import build_temporal_dataset
from src.robust_validation import (
    DEFAULT_SEEDS,
    aggregate_confusion_stability,
    aggregate_metric_summary,
    aggregate_per_class_stability,
    run_repeated_validation,
)

RESULT_ROOT = PROJECT_ROOT / 'results'
FIGURE_ROOT = RESULT_ROOT / 'figures'
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
CLASS_LABELS = list(FIRST_VERSION_CLASSES)
WINDOWS_S = (30, 60, 90, 120)
TARGET_CLASSES = ['FLB', 'LLB', 'LOCAC', 'RW', 'RI', 'SGBTR', 'LOCA']


## Cohort and protocol

The strict cohort requires a known first protection time and an actual last sampled point before that time. The primary matched cohort is the intersection of the 30/60/90/120 s strict cohorts. Unknown first protection times never enter this evaluation.

In [2]:
feature_groups, removed_features = input_feature_groups()
assert len(feature_groups['A_strict_38']) == 38
assert len(feature_groups['B_without_potential_leakage']) == 37
assert len(feature_groups['C_without_SLBIC_initial']) == 24

dataset, window_points = build_temporal_dataset(
    project_root=PROJECT_ROOT, windows_s=WINDOWS_S, classes=tuple(CLASS_LABELS)
)
assert dataset[['sample_id', 'window_s']].duplicated().sum() == 0
assert set(dataset['window_s']) == set(WINDOWS_S)

strict = dataset.loc[
    dataset['strict_pre_protection'].astype(bool) & dataset['first_protection_s'].notna()
].copy()
assert strict['first_protection_s'].notna().all()
assert (strict['first_protection_s'] > strict['last_sample_s']).all()

cohort_ids = set.intersection(
    *(set(strict.loc[strict['window_s'].eq(window), 'sample_id']) for window in WINDOWS_S)
)
matched = strict.loc[strict['sample_id'].isin(cohort_ids)].copy()
assert len(cohort_ids) == matched['sample_id'].nunique()
assert matched.groupby('sample_id')['window_s'].nunique().eq(len(WINDOWS_S)).all()

unsupported_classes = [label for label in CLASS_LABELS if label not in set(matched['accident_class'])]
assert set(unsupported_classes) == {'LLB', 'LR', 'MD', 'SLBOC'}
assert matched['accident_class'].value_counts().min() >= 3
cohort_counts = (
    matched.groupby(['window_s', 'accident_class'], as_index=False)['sample_id']
    .nunique()
)
print('Matched strict trajectories:', len(cohort_ids))
print('Unknown first_protection_s excluded from strict:', int(dataset['first_protection_s'].isna().sum()))
print('Strict classes with no evaluable trajectory:', unsupported_classes)
display(cohort_counts)


Matched strict trajectories: 471
Unknown first_protection_s excluded from strict: 2186
Strict classes with no evaluable trajectory: ['LLB', 'LR', 'MD', 'SLBOC']


,window_s,accident_class,sample_id
0,30,FLB,99
1,30,LOCA,76
2,30,LOCAC,79
3,30,RI,28
4,30,RW,3
5,30,SGATR,44
6,30,SGBTR,54
7,30,SLBIC,88
8,60,FLB,99
9,60,LOCA,76


## Repeated grouped validation

Ten fixed seeds create 60/20/20 train/validation/test splits from unique trajectory IDs. Validation selects one model and training mode per window, feature group, and split; test is never used for selection. Raw training is compared with train-only class balancing.

In [3]:
SEEDS = DEFAULT_SEEDS
run = run_repeated_validation(
    matched,
    feature_groups,
    CLASS_LABELS,
    windows_s=WINDOWS_S,
    seeds=SEEDS,
    scope='strict_pre_protection_only',
    cohort='strict_matched_30_60_90_120',
)
metrics = run['metrics']
per_class = run['per_class']
confusion = run['confusion']
support_audit = run['support_audit']
assignments = run['assignments']

assert len(SEEDS) == 10
assert metrics['split_index'].nunique() == len(SEEDS)
assert set(metrics['eval_split']) == {'validation', 'test'}
assert set(metrics['model']) == {'logistic_regression', 'random_forest', 'hist_gradient_boosting'}
assert set(metrics['training_mode']) == {'raw', 'class_balanced'}
assert metrics[['accuracy', 'macro_f1_fixed_12', 'macro_f1_observed_classes', 'balanced_accuracy']].notna().all().all()
assert assignments.groupby(['sample_id', 'split_index'])['partition'].nunique().eq(1).all()
assert support_audit['test_support_lt_3'].sum() > 0
print('Metrics rows:', len(metrics))
print('Per-class rows:', len(per_class))
print('Confusion rows:', len(confusion))
display(metrics.loc[(metrics['eval_split'].eq('test')) & metrics['validation_selected'], [
    'window_s', 'input_group', 'training_mode', 'model',
    'macro_f1_fixed_12', 'macro_f1_observed_classes', 'balanced_accuracy'
]].head(20))


Metrics rows: 1440
Per-class rows: 8640
Confusion rows: 103680


,window_s,input_group,training_mode,model,macro_f1_fixed_12,macro_f1_observed_classes,balanced_accuracy
5,30,A_strict_38,raw,hist_gradient_boosting,0.219565,0.376398,0.444444
21,30,B_without_potential_leakage,class_balanced,random_forest,0.219565,0.376398,0.444444
33,30,C_without_SLBIC_initial,class_balanced,random_forest,0.219565,0.376398,0.444444
45,60,A_strict_38,class_balanced,random_forest,0.219565,0.376398,0.444444
57,60,B_without_potential_leakage,class_balanced,random_forest,0.219565,0.376398,0.444444
67,60,C_without_SLBIC_initial,class_balanced,logistic_regression,0.219565,0.376398,0.444444
81,90,A_strict_38,class_balanced,random_forest,0.219565,0.376398,0.444444
93,90,B_without_potential_leakage,class_balanced,random_forest,0.219565,0.376398,0.444444
105,90,C_without_SLBIC_initial,class_balanced,random_forest,0.219565,0.376398,0.444444
119,120,A_strict_38,class_balanced,hist_gradient_boosting,0.578393,0.991531,0.991071


## Split summaries, support audit, and confusion stability

The 95% intervals below use a normal approximation to the standard error across the ten repeated test splits. Fixed-12 Macro-F1 includes all 12 audit classes; observed-class Macro-F1 averages only classes with positive test support in that split. The support-threshold Macro-F1 values exclude classes below the named threshold.

In [4]:
metric_summary = aggregate_metric_summary(metrics)
selected_test = metrics.loc[(metrics['eval_split'].eq('test')) & metrics['validation_selected']].copy()
selected_metric_summary = aggregate_metric_summary(
    selected_test,
    ['cohort', 'scope', 'window_s', 'input_group'],
)

selected_per_class = per_class.loc[per_class['validation_selected']].copy()
per_class_stability = aggregate_per_class_stability(
    selected_per_class,
    ['cohort', 'scope', 'window_s', 'input_group', 'accident_class'],
)

selected_confusion = confusion.loc[confusion['validation_selected']].copy()
confusion_stability = aggregate_confusion_stability(
    selected_confusion,
    CLASS_LABELS,
    ['cohort', 'scope', 'window_s', 'input_group'],
)

assert len(selected_test) == len(SEEDS) * len(WINDOWS_S) * len(feature_groups)
assert set(per_class_stability.loc[per_class_stability['accident_class'].isin(unsupported_classes), 'status']) == {'not_evaluable_no_strict_trajectory'}
assert per_class.loc[per_class['cohort_support'].eq(0), 'recall'].isna().all()
assert confusion_stability['n_splits'].eq(len(SEEDS)).all()

display(selected_metric_summary.loc[selected_metric_summary['input_group'].eq('A_strict_38'), [
    'window_s', 'accuracy_mean', 'macro_f1_fixed_12_mean', 'macro_f1_fixed_12_std',
    'macro_f1_fixed_12_ci95_low', 'macro_f1_fixed_12_ci95_high',
    'macro_f1_observed_classes_mean', 'macro_f1_observed_classes_std',
    'balanced_accuracy_mean', 'balanced_accuracy_std'
]].sort_values('window_s'))
display(per_class_stability.loc[
    per_class_stability['input_group'].eq('A_strict_38') & per_class_stability['window_s'].eq(120)
].sort_values('accident_class'))


,window_s,accuracy_mean,macro_f1_fixed_12_mean,macro_f1_fixed_12_std,macro_f1_fixed_12_ci95_low,macro_f1_fixed_12_ci95_high,macro_f1_observed_classes_mean,macro_f1_observed_classes_std,balanced_accuracy_mean,balanced_accuracy_std
0,30,0.441053,0.219617,0.011655,0.212393,0.226841,0.376487,0.019980,0.447619,0.011348
3,60,0.445263,0.220200,0.010441,0.213729,0.226672,0.377486,0.017899,0.447619,0.011348
6,90,0.445263,0.220217,0.011674,0.212982,0.227453,0.377515,0.020012,0.447619,0.011348
9,120,0.990526,0.578359,0.004247,0.575727,0.580992,0.991473,0.007281,0.988810,0.010950


,cohort,scope,window_s,input_group,accident_class,cohort_support,n_splits,n_evaluable_splits,support_mean,support_min,...,status,reliable_for_gate,recall_n,recall_mean,recall_std,recall_median,recall_min,recall_max,recall_ci95_low,recall_ci95_high
108,strict_matched_30_60_90_120,strict_pre_protection_only,120,A_strict_38,FLB,99,10,10,20.0,20,...,evaluable,True,10,1.000000,0.000000,1.0,1.000000,1.0,1.000000,1.000000
109,strict_matched_30_60_90_120,strict_pre_protection_only,120,A_strict_38,LLB,0,10,0,0.0,0,...,not_evaluable_no_strict_trajectory,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
110,strict_matched_30_60_90_120,strict_pre_protection_only,120,A_strict_38,LOCA,76,10,10,15.0,15,...,evaluable,True,10,0.980000,0.044997,1.0,0.866667,1.0,0.952111,1.007889
111,strict_matched_30_60_90_120,strict_pre_protection_only,120,A_strict_38,LOCAC,79,10,10,16.0,16,...,evaluable,True,10,0.975000,0.032275,1.0,0.937500,1.0,0.954996,0.995004
112,strict_matched_30_60_90_120,strict_pre_protection_only,120,A_strict_38,LR,0,10,0,0.0,0,...,not_evaluable_no_strict_trajectory,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,strict_matched_30_60_90_120,strict_pre_protection_only,120,A_strict_38,MD,0,10,0,0.0,0,...,not_evaluable_no_strict_trajectory,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,strict_matched_30_60_90_120,strict_pre_protection_only,120,A_strict_38,RI,28,10,10,6.0,6,...,evaluable,True,10,0.966667,0.070273,1.0,0.833333,1.0,0.923111,1.010222
115,strict_matched_30_60_90_120,strict_pre_protection_only,120,A_strict_38,RW,3,10,0,0.0,0,...,low_support,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
116,strict_matched_30_60_90_120,strict_pre_protection_only,120,A_strict_38,SGATR,44,10,10,9.0,9,...,evaluable,True,10,1.000000,0.000000,1.0,1.000000,1.0,1.000000,1.000000
117,strict_matched_30_60_90_120,strict_pre_protection_only,120,A_strict_38,SGBTR,54,10,10,11.0,11,...,evaluable,True,10,1.000000,0.000000,1.0,1.000000,1.0,1.000000,1.000000


## A/B/C sensitivity and gate decision

A is the strict 38-variable group, B removes LVCR, and C removes the 14 SLBIC initial-condition candidates. The gate requires a meaningful 120 s advantage over both 60 s and 90 s, acceptable split variation, and at least five test examples for every evaluable target class in every repeat.

In [5]:
metric_names = ['accuracy', 'macro_f1_fixed_12', 'macro_f1_observed_classes', 'balanced_accuracy']
selected_120 = selected_test.loc[selected_test['window_s'].eq(120)].copy()
base_keys = ['cohort', 'scope', 'split_index', 'seed', 'window_s']
base = selected_120.loc[selected_120['input_group'].eq('A_strict_38'), base_keys + metric_names].rename(
    columns={name: f'A_{name}' for name in metric_names}
)
sensitivity_parts = []
for group_name in feature_groups:
    current = selected_120.loc[selected_120['input_group'].eq(group_name), base_keys + metric_names + ['model', 'training_mode']].rename(
        columns={name: f'group_{name}' for name in metric_names}
    )
    current = current.merge(base, on=base_keys, how='inner', validate='one_to_one')
    current['input_group'] = group_name
    for name in metric_names:
        current[f'delta_{name}_vs_A'] = current[f'group_{name}'] - current[f'A_{name}']
    sensitivity_parts.append(current)
sensitivity = pd.concat(sensitivity_parts, ignore_index=True)
sensitivity_summary = sensitivity.groupby('input_group', as_index=False).agg(
    fixed_mean=('group_macro_f1_fixed_12', 'mean'),
    fixed_std=('group_macro_f1_fixed_12', 'std'),
    observed_mean=('group_macro_f1_observed_classes', 'mean'),
    observed_std=('group_macro_f1_observed_classes', 'std'),
    balanced_accuracy_mean=('group_balanced_accuracy', 'mean'),
    balanced_accuracy_std=('group_balanced_accuracy', 'std'),
    delta_fixed_mean=('delta_macro_f1_fixed_12_vs_A', 'mean'),
    delta_observed_mean=('delta_macro_f1_observed_classes_vs_A', 'mean'),
)

primary = selected_metric_summary.loc[selected_metric_summary['input_group'].eq('A_strict_38')].set_index('window_s')
f60, f90, f120 = (primary.loc[window] for window in (60, 90, 120))
comparison_ok = bool(
    f120['macro_f1_fixed_12_mean'] > max(f60['macro_f1_fixed_12_mean'], f90['macro_f1_fixed_12_mean']) + 0.10
    and f120['macro_f1_observed_classes_mean'] > max(f60['macro_f1_observed_classes_mean'], f90['macro_f1_observed_classes_mean']) + 0.10
)
variance_ok = bool(
    f120['macro_f1_fixed_12_std'] <= 0.10
    and f120['macro_f1_observed_classes_std'] <= 0.10
)
target_recall = per_class_stability.loc[
    (per_class_stability['input_group'].eq('A_strict_38'))
    & (per_class_stability['window_s'].eq(120))
    & (per_class_stability['accident_class'].isin(TARGET_CLASSES))
].copy()
evaluable_target_recall = target_recall.loc[target_recall['cohort_support'].gt(0)]
support_ok = bool(
    not evaluable_target_recall.empty
    and evaluable_target_recall['reliable_for_gate'].all()
    and evaluable_target_recall['n_splits_support_ge_5'].eq(len(SEEDS)).all()
)
recall_ok = bool(
    not evaluable_target_recall.empty
    and evaluable_target_recall['recall_std'].notna().all()
    and evaluable_target_recall['recall_std'].le(0.20).all()
    and evaluable_target_recall['recall_n'].eq(len(SEEDS)).all()
)
gate_pass = bool(comparison_ok and variance_ok and support_ok and recall_ok)
gate_reasons = {
    'matched_120_advantage_over_60_and_90': comparison_ok,
    '120_split_variation_within_0.10_std': variance_ok,
    'all_evaluable_target_classes_have_test_support_ge_5': support_ok,
    'evaluable_target_recall_std_within_0.20': recall_ok,
}
print('Gate decision:', 'A_ENTER_SEQUENCE_MODELS' if gate_pass else 'B_REQUIRE_STRICTER_DATA_OR_EXTERNAL_VALIDATION')
print('Gate checks:', gate_reasons)
display(sensitivity_summary)


Gate decision: B_REQUIRE_STRICTER_DATA_OR_EXTERNAL_VALIDATION
Gate checks: {'matched_120_advantage_over_60_and_90': True, '120_split_variation_within_0.10_std': True, 'all_evaluable_target_classes_have_test_support_ge_5': False, 'evaluable_target_recall_std_within_0.20': False}


,input_group,fixed_mean,fixed_std,observed_mean,observed_std,balanced_accuracy_mean,balanced_accuracy_std,delta_fixed_mean,delta_observed_mean
0,A_strict_38,0.578359,0.004247,0.991473,0.007281,0.988810,0.010950,0.000000,0.000000
1,B_without_potential_leakage,0.578359,0.004247,0.991473,0.007281,0.988810,0.010950,0.000000,0.000000
2,C_without_SLBIC_initial,0.578391,0.004465,0.991527,0.007654,0.988333,0.011423,0.000031,0.000054


## Persist audit tables and figures

In [6]:
def json_safe(value):
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    return value

metrics.to_csv(RESULT_ROOT / '12_robust_validation_metrics.csv', index=False)
metric_summary.to_csv(RESULT_ROOT / '12_robust_validation_metric_summary.csv', index=False)
selected_metric_summary.to_csv(RESULT_ROOT / '12_matched_cohort_comparison.csv', index=False)
per_class_stability.to_csv(RESULT_ROOT / '12_per_class_stability.csv', index=False)
confusion_stability.to_csv(RESULT_ROOT / '12_confusion_stability.csv', index=False)
support_audit.to_csv(RESULT_ROOT / '12_support_audit.csv', index=False)
sensitivity.to_csv(RESULT_ROOT / '12_sensitivity_comparison.csv', index=False)
sensitivity_summary.to_csv(RESULT_ROOT / '12_sensitivity_summary.csv', index=False)
assignments.to_csv(RESULT_ROOT / '12_split_assignments.csv', index=False)

top_confusions = confusion_stability.loc[
    (confusion_stability['input_group'].eq('A_strict_38'))
    & (confusion_stability['window_s'].eq(120))
    & (confusion_stability['pair_count_mean'].gt(0))
].sort_values(['pair_frequency', 'pair_count_mean'], ascending=False).head(20)
summary = {
    'experiment': 'repeated robust validation of strict pre-protection temporal diagnosability',
    'classes_fixed_12': CLASS_LABELS,
    'strict_classes_with_no_known_protection': unsupported_classes,
    'windows_s': list(WINDOWS_S),
    'matched_cohort_count': int(len(cohort_ids)),
    'cohort_counts': cohort_counts.to_dict(orient='records'),
    'seeds': [int(seed) for seed in SEEDS],
    'split_protocol': '10 repeated 60/20/20 stratified splits on unique sample_id; same assignments reused across windows and feature groups',
    'model_protocol': 'fixed Logistic Regression, Random Forest, HistGradientBoosting; raw and train-only class-balanced modes; validation-only selection',
    'ci_protocol': 'normal approximation mean +/- 1.96*sample_std/sqrt(n) across repeated test splits',
    'selected_metric_summary': selected_metric_summary.to_dict(orient='records'),
    'selected_per_class_stability': per_class_stability.to_dict(orient='records'),
    'top_120s_confusion_pairs_group_A': top_confusions.to_dict(orient='records'),
    'sensitivity_summary_120s': sensitivity_summary.to_dict(orient='records'),
    'gate': {
        'decision': 'A_ENTER_SEQUENCE_MODELS' if gate_pass else 'B_REQUIRE_STRICTER_DATA_OR_EXTERNAL_VALIDATION',
        'checks': gate_reasons,
        'target_recall_used_for_gate': target_recall.to_dict(orient='records'),
        'note': 'Not-evaluable classes are excluded from recall reliability claims; fixed-12 Macro-F1 retains them as zero-score labels for a conservative audit metric.'
    }
}
with open(RESULT_ROOT / '12_robust_validation_summary.json', 'w', encoding='utf-8') as handle:
    json.dump(json_safe(summary), handle, ensure_ascii=False, indent=2)

assert (RESULT_ROOT / '12_robust_validation_metrics.csv').exists()
assert (RESULT_ROOT / '12_per_class_stability.csv').exists()
assert (RESULT_ROOT / '12_confusion_stability.csv').exists()
assert (RESULT_ROOT / '12_support_audit.csv').exists()
assert (RESULT_ROOT / '12_robust_validation_summary.json').exists()


In [7]:
primary_selected = selected_test.loc[selected_test['input_group'].eq('A_strict_38')]
fig, ax = plt.subplots(figsize=(10, 5))
fixed_data = [primary_selected.loc[primary_selected['window_s'].eq(window), 'macro_f1_fixed_12'] for window in WINDOWS_S]
observed_data = [primary_selected.loc[primary_selected['window_s'].eq(window), 'macro_f1_observed_classes'] for window in WINDOWS_S]
positions = np.arange(len(WINDOWS_S))
ax.boxplot(fixed_data, positions=positions - 0.18, widths=0.30, patch_artist=True, boxprops={'facecolor': '#4472C4'}, medianprops={'color': 'white'}, tick_labels=[str(window) for window in WINDOWS_S])
ax.boxplot(observed_data, positions=positions + 0.18, widths=0.30, patch_artist=True, boxprops={'facecolor': '#ED7D31'}, medianprops={'color': 'white'}, tick_labels=[str(window) for window in WINDOWS_S])
ax.scatter([], [], color='#4472C4', label='Fixed-12 Macro-F1')
ax.scatter([], [], color='#ED7D31', label='Observed-class Macro-F1')
ax.set_xlabel('Window (s)')
ax.set_ylabel('Test Macro-F1')
ax.set_title('Repeated test Macro-F1 distribution on matched strict cohort')
ax.set_ylim(-0.02, 1.02)
ax.grid(axis='y', alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_ROOT / '12_macro_f1_distribution_by_window.png', dpi=150)
plt.close(fig)

frame = selected_metric_summary.loc[selected_metric_summary['input_group'].eq('A_strict_38')].sort_values('window_s')
fig, ax = plt.subplots(figsize=(10, 5))
for metric, label, color in [('macro_f1_fixed_12', 'Fixed-12 Macro-F1', '#4472C4'), ('macro_f1_observed_classes', 'Observed-class Macro-F1', '#ED7D31'), ('balanced_accuracy', 'Balanced Accuracy', '#70AD47')]:
    lower = frame[f'{metric}_mean'] - frame[f'{metric}_ci95_low']
    upper = frame[f'{metric}_ci95_high'] - frame[f'{metric}_mean']
    ax.errorbar(frame['window_s'], frame[f'{metric}_mean'], yerr=[lower, upper], marker='o', capsize=4, label=label, color=color)
ax.set_xticks(WINDOWS_S)
ax.set_ylim(-0.02, 1.02)
ax.set_xlabel('Window (s)')
ax.set_ylabel('Mean test score with 95% CI')
ax.set_title('Matched-cohort repeated validation comparison')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_ROOT / '12_matched_cohort_comparison.png', dpi=150)
plt.close(fig)


In [8]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharey=True)
plot_labels = [label for label in CLASS_LABELS if label not in unsupported_classes]
for ax, window in zip(axes.flat, WINDOWS_S):
    frame = per_class_stability.loc[
        (per_class_stability['input_group'].eq('A_strict_38'))
        & (per_class_stability['window_s'].eq(window))
        & (per_class_stability['cohort_support'].gt(0))
    ].set_index('accident_class').reindex(plot_labels).reset_index()
    frame = frame.loc[frame['recall_mean'].notna()]
    x = np.arange(len(frame))
    lower = frame['recall_mean'] - frame['recall_ci95_low']
    upper = frame['recall_ci95_high'] - frame['recall_mean']
    ax.errorbar(x, frame['recall_mean'], yerr=[lower, upper], fmt='o', capsize=4, color='#4472C4')
    ax.set_xticks(x, frame['accident_class'], rotation=55, ha='right')
    ax.set_title(f'{window} s')
    ax.set_ylim(-0.02, 1.02)
    ax.grid(axis='y', alpha=0.25)
axes[0, 0].set_ylabel('Recall mean +/- 95% CI')
axes[1, 0].set_ylabel('Recall mean +/- 95% CI')
fig.suptitle('Per-class recall stability on validation-selected test models')
fig.tight_layout()
fig.savefig(FIGURE_ROOT / '12_per_class_recall_stability.png', dpi=150)
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharey=True)
for ax, window in zip(axes.flat, WINDOWS_S):
    frame = support_audit.loc[support_audit['window_s'].eq(window)]
    data = [frame.loc[frame['accident_class'].eq(label), 'test_support'] for label in CLASS_LABELS]
    ax.boxplot(data, tick_labels=CLASS_LABELS, showmeans=True)
    ax.axhline(3, color='#ED7D31', linestyle='--', linewidth=1, label='support = 3')
    ax.axhline(5, color='#70AD47', linestyle='--', linewidth=1, label='support = 5')
    ax.set_title(f'{window} s')
    ax.tick_params(axis='x', rotation=55)
    ax.grid(axis='y', alpha=0.25)
axes[0, 0].set_ylabel('Test support per split')
axes[1, 0].set_ylabel('Test support per split')
axes[0, 0].legend(fontsize=8)
fig.suptitle('Minimum-support audit across repeated splits')
fig.tight_layout()
fig.savefig(FIGURE_ROOT / '12_support_distribution.png', dpi=150)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
sensitivity_plot = sensitivity_summary.set_index('input_group').reindex(list(feature_groups))
x = np.arange(len(sensitivity_plot))
width = 0.35
ax.bar(x - width / 2, sensitivity_plot['fixed_mean'], width, label='Fixed-12 Macro-F1', color='#4472C4')
ax.bar(x + width / 2, sensitivity_plot['observed_mean'], width, label='Observed-class Macro-F1', color='#ED7D31')
ax.set_xticks(x, sensitivity_plot.index, rotation=20)
ax.set_ylim(-0.02, 1.02)
ax.set_ylabel('120 s selected test mean')
ax.set_title('A/B/C sensitivity comparison')
ax.grid(axis='y', alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_ROOT / '12_sensitivity_comparison.png', dpi=150)
plt.close(fig)


In [9]:
for figure_name in [
    '12_macro_f1_distribution_by_window.png',
    '12_matched_cohort_comparison.png',
    '12_per_class_recall_stability.png',
    '12_support_distribution.png',
    '12_sensitivity_comparison.png',
]:
    assert (FIGURE_ROOT / figure_name).exists()

assert (RESULT_ROOT / '12_robust_validation_metrics.csv').exists()
assert (RESULT_ROOT / '12_robust_validation_metric_summary.csv').exists()
assert (RESULT_ROOT / '12_matched_cohort_comparison.csv').exists()
assert (RESULT_ROOT / '12_per_class_stability.csv').exists()
assert (RESULT_ROOT / '12_confusion_stability.csv').exists()
assert (RESULT_ROOT / '12_support_audit.csv').exists()
assert (RESULT_ROOT / '12_sensitivity_comparison.csv').exists()
assert (RESULT_ROOT / '12_robust_validation_summary.json').exists()
print('Selected 60/90/120 s A-group summary:')
display(selected_metric_summary.loc[selected_metric_summary['input_group'].eq('A_strict_38')].sort_values('window_s'))
print('Top stable 120 s A-group confusion pairs:')
display(top_confusions[['pair', 'pair_frequency', 'split_count_present', 'pair_count_mean', 'forward_count_mean', 'reverse_count_mean']])
print('FULL ROBUST VALIDATION PASSED')


Selected 60/90/120 s A-group summary:


,cohort,scope,window_s,input_group,accuracy_n,accuracy_mean,accuracy_std,accuracy_median,accuracy_min,accuracy_max,...,macro_f1_support_ge_5_ci95_low,macro_f1_support_ge_5_ci95_high,balanced_accuracy_n,balanced_accuracy_mean,balanced_accuracy_std,balanced_accuracy_median,balanced_accuracy_min,balanced_accuracy_max,balanced_accuracy_ci95_low,balanced_accuracy_ci95_high
0,strict_matched_30_60_90_120,strict_pre_protection_only,30,A_strict_38,10,0.441053,0.037937,0.452632,0.357895,0.473684,...,0.364103,0.388870,10,0.447619,0.011348,0.448413,0.428571,0.460317,0.440586,0.454653
3,strict_matched_30_60_90_120,strict_pre_protection_only,60,A_strict_38,10,0.445263,0.033673,0.452632,0.357895,0.473684,...,0.366392,0.388580,10,0.447619,0.011348,0.448413,0.428571,0.460317,0.440586,0.454653
6,strict_matched_30_60_90_120,strict_pre_protection_only,90,A_strict_38,10,0.445263,0.037480,0.452632,0.347368,0.473684,...,0.365112,0.389919,10,0.447619,0.011348,0.448413,0.428571,0.460317,0.440586,0.454653
9,strict_matched_30_60_90_120,strict_pre_protection_only,120,A_strict_38,10,0.990526,0.007767,0.989474,0.978947,1.000000,...,0.986960,0.995986,10,0.988810,0.010950,0.991071,0.967262,1.000000,0.982023,0.995596


Top stable 120 s A-group confusion pairs:


,pair,pair_frequency,split_count_present,pair_count_mean,forward_count_mean,reverse_count_mean
630,LOCAC↔SLBIC,0.4,4,0.4,0.4,0.0
595,FLB↔LOCA,0.2,2,0.3,0.0,0.3
645,RI↔RW,0.2,2,0.2,0.2,0.0


FULL ROBUST VALIDATION PASSED
